# 03 · Construir el subconjunto curado con percentiles · Resolución

Solución comentada de esta práctica de livecoding.

Seleccioná el kernel de `.venv` y usá **Reiniciar kernel y ejecutar todo**. La sección 3 contiene la función resuelta y la sección 4 ejecuta las pruebas de la práctica junto con la comprobación sobre datos reales. El resultado completo queda en la variable `resultado`.

La carga reutiliza `practicas_livecoding/preparacion.py` y los datasets ya descargados en `practicas_livecoding/datos/`. Conservá ambas carpetas dentro del mismo proyecto.

## 1. Preparación lista

La base equivale a `df` del notebook original: dormitorios imputados, marca de imputación y tres ratios. No se aplicó el filtro de curado.

Los archivos de `practicas_livecoding/datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_viviendas()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 20,640 | Columnas: 14


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_imputados,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,False,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,False,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,False,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,False,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,False,6.281853,0.172096,2.181467


## 2. Consigna y resultado esperado

Devolvé las filas que cumplan **todas** estas condiciones:

1. `median_house_value` es estrictamente menor que el máximo de la entrada.
2. `rooms_per_household`, `bedrooms_per_room` y `population_per_household`
   son menores o iguales a sus respectivos percentiles.

Calculá cada corte sobre **toda la entrada, antes de filtrar**, con
`quantile(percentil)` y su interpolación predeterminada. El valor predeterminado corresponde al P99.
Conservá columnas, índices y orden original; no agregues columnas ni modifiques la entrada.
Una entrada vacía devuelve una copia vacía. Podés asumir `0 <= percentil <= 1`.

**Atención:** usar el percentil de un solo ratio, filtrar con OR o recalcular
los cortes después de excluir el precio máximo produce otro resultado.

## 3. Resolución comentada

1. `quantile(percentil)` produce un corte por cada ratio, usando toda la entrada.
2. `le(cortes)` compara cada columna con su propio corte; `all(axis=1)` exige cumplir los tres.
3. La máscara final combina esa condición con un precio estrictamente menor que el máximo.

`loc` conserva el orden y los índices de las filas. Recalcular los percentiles después
de otro filtro cambiaría los cortes y no cumpliría la consigna.

In [3]:
def curar_viviendas(datos, percentil=0.99):
    """Conserva las filas sin precio máximo y con los tres ratios dentro del corte."""
    ratios = [
        "rooms_per_household",
        "bedrooms_per_room",
        "population_per_household",
    ]

    # Los cortes se calculan sobre toda la entrada, antes de excluir filas.
    cortes = datos[ratios].quantile(percentil)
    sin_tope = datos["median_house_value"].lt(datos["median_house_value"].max())
    ratios_en_rango = datos[ratios].le(cortes).all(axis=1)

    # La máscara conserva el índice y el orden originales.
    return datos.loc[sin_tope & ratios_en_rango].copy()

## 4. Pruebas y ejecución real

Se conservan los casos y las comprobaciones de la práctica original. Los mensajes `OK` confirman que pasan y que la entrada permanece intacta. La tabla muestra las primeras 20 filas; `resultado` contiene la salida completa.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "median_house_value": [10, 20, 30, 40, 50, 60, 100],
        "rooms_per_household": [1, 2, 3, 4, 5, 6, 7],
        "bedrooms_per_room": [7, 6, 5, 4, 3, 2, 1],
        "population_per_household": [1, 1, 1, 1, 1, 9, 1],
        "dato_extra": list("abcdefg"),
    }, index=[9, 8, 7, 6, 5, 4, 3])
    original = caso.copy(deep=True)

    comparar(funcion(caso, percentil=0.75), caso.iloc[[2, 3, 4]])
    comparar(funcion(caso, percentil=0.5), caso.iloc[[3]])
    comparar(funcion(caso, percentil=1), caso.iloc[:6])
    comparar(funcion(caso), caso.iloc[[1, 2, 3, 4]])

    todos_en_maximo = caso.assign(median_house_value=100)
    comparar(funcion(todos_en_maximo), todos_en_maximo.iloc[:0])
    comparar(funcion(caso.iloc[:0]), caso.iloc[:0])
    comparar(caso, original)
    print("OK: tres ratios, percentiles sobre la entrada completa, máximo estricto y orden original.")


probar(curar_viviendas)

# Comprobamos también que el dataset real permanezca intacto.
antes = datos.copy(deep=True)
resultado = curar_viviendas(datos)
comparar(datos, antes)
ratios = ["rooms_per_household", "bedrooms_per_room", "population_per_household"]
assert list(resultado.columns) == list(datos.columns)
assert resultado["median_house_value"].lt(datos["median_house_value"].max()).all()
assert resultado[ratios].le(datos[ratios].quantile(0.99)).all().all()
assert resultado.index.isin(datos.index).all()
assert 0 < len(resultado) < len(datos)
print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
display(resultado.head(20))

OK: tres ratios, percentiles sobre la entrada completa, máximo estricto y orden original.
OK: dataset real. Resultado: 19,087 filas.


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_imputados,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,False,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,False,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,False,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,False,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,False,6.281853,0.172096,2.181467
5,-122.25,37.85,52.0,919.0,213.0,413.0,193.0,4.0368,269700.0,NEAR BAY,False,4.761658,0.231774,2.139896
6,-122.25,37.84,52.0,2535.0,489.0,1094.0,514.0,3.6591,299200.0,NEAR BAY,False,4.931907,0.192899,2.128405
7,-122.25,37.84,52.0,3104.0,687.0,1157.0,647.0,3.1200,241400.0,NEAR BAY,False,4.797527,0.221327,1.788253
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY,False,4.294118,0.260274,2.026891
9,-122.25,37.84,52.0,3549.0,707.0,1551.0,714.0,3.6912,261100.0,NEAR BAY,False,4.970588,0.199211,2.172269


## Para repasar

Volvé a la práctica sin resolver e intentá escribir la función desde cero. Explicá qué representa cada fila de la salida, cómo se resuelven los casos límite y por qué el cálculo respeta la consigna.